# annotation の画像+マスク確認（都度読み込み）

`geometry_uid`（または先頭数桁）や `dataset_id` を指定して、該当する annotation の
元DICOM画像とマスクをその場で読み込み、matplotlibで重畳表示するnotebook。

dashboard.html にプレビュー機能を組み込む案は、データが増え続けるとスケールしない
ため見送った。代わりにこちらは**指定した1件だけをその都度読み込む**ので、
事前生成もサーバーも不要。`issues.json` / `selection_decisions.json` があれば、
その annotation の検出結果・採否もあわせて表示する（無ければ空欄になるだけで動く）。

## 0. 前提

カーネルは `notebooks/pipeline.ipynb` と同じ、このリポジトリの `uv` 環境
（`segmentation-validation` カーネル）に紐づいていること。

In [ ]:
import os

PROJECT_ROOT = "/mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation"
os.chdir(PROJECT_ROOT)
print("cwd:", os.getcwd())

## 1. データを読み込む（重い処理はここだけ）

元JSONをパースして annotation 一覧を作る（`scan`/`check`とは独立に動く。
DICOM/マスクの画素はまだ読まない）。

In [ ]:
%matplotlib inline

from segmentation_validation.config import load_config
from segmentation_validation.cli import _load_context, _fingerprint

config = load_config(None, [])
ctx = _load_context(config)
print(f"annotation数: {len(ctx.records)} (対象外 {len(ctx.out_of_scope)})")

`check` / `select` を既に実行済みなら、その結果（検出されたcheck・最終的な採否）も
読み込んでおく。まだ無くても以降のセルは動く（検出結果・採否の欄が空になるだけ）。

In [ ]:
from segmentation_validation.cli import _read_issues, _read_selection, _read_image_decisions

out_dir = config.validation_dir / _fingerprint(config)
issues = _read_issues(out_dir / "issues.json") if (out_dir / "issues.json").exists() else []
decisions = _read_selection(out_dir / "selection_decisions.json")
image_decisions = _read_image_decisions(out_dir / "image_decisions.json")

print(f"fingerprint: {_fingerprint(config)}")
print(f"issues: {len(issues)} / selection_decisions: {len(decisions)} / image_decisions: {len(image_decisions)}")

## 2. 検索・表示ヘルパー

- `find_records(query, dataset_id=None)`: `geometry_uid` の完全一致 → 前方一致の順で探す。
  `query=None` なら `dataset_id` だけで絞り込む（一覧表示用）
- `show_case(record)`: DICOM画像にマスクを重ねて表示し、所在・検出結果・採否を出す

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from segmentation_validation.core.imageio import load_dicom_image, load_binary_mask

_pool = list(ctx.records) + list(ctx.out_of_scope)


def find_records(query, dataset_id=None):
    """geometry_uidの完全一致→前方一致で探す。queryがNoneならdataset_idだけで絞る。"""
    pool = _pool
    if dataset_id:
        pool = [r for r in pool if r.dataset_id == dataset_id]
    if not query:
        return pool
    exact = [r for r in pool if r.geometry_uid == query]
    if exact:
        return exact
    return [r for r in pool if r.geometry_uid.startswith(query)]


def show_case(record, mask_color=(1.0, 0.15, 0.15), mask_alpha=0.4, figsize=(7, 7)):
    """DICOM画像にマスクを重畳表示し、所在・検出結果・採否をあわせて出す。"""
    image = load_dicom_image(record.resolved_image_path)
    mask = (
        load_binary_mask(record.resolved_path_mask)
        if record.resolved_path_mask is not None
        else None
    )

    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image, cmap="gray")
    if mask is not None:
        overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
        overlay[mask] = (*mask_color, mask_alpha)
        ax.imshow(overlay)
    ax.set_title(
        f"{record.short_uid}  {record.dataset_id}  {record.institution}/{record.patient_id}"
    )
    ax.axis("off")
    plt.show()

    print("geometry_uid  :", record.geometry_uid)
    print("dataset_id    :", record.dataset_id, "  source_json:", record.source_json)
    print("institution   :", record.institution)
    print("patient/study :", record.patient_id, "/", record.study)
    print("file/series   :", record.file, "/", record.series)
    print("annotator     :", record.user, " ", record.timestamp)
    print("labels        :", ", ".join(record.label_lines()))
    print("image_path    :", record.resolved_image_path)
    print("mask_path     :", record.resolved_path_mask)

    own_issues = [i for i in issues if i.geometry_uid == record.geometry_uid]
    if own_issues:
        print("\n検出されたcheck:")
        for i in own_issues:
            print(f"  [{i.severity.value:7s}] {i.check_id:32s} {i.message}")
    else:
        print("\n検出されたcheck: なし")

    decision = next(
        (d for d in decisions if d.geometry_uid == record.geometry_uid), None
    )
    if decision is not None:
        print(
            f"\n採否: {decision.final_decision.value}"
            f"  (reason={decision.reason}, source={decision.decision_source.value})"
        )
    else:
        print("\n採否: selection_decisions.json に該当なし（select 未実行、または対象外）")

    return record

## 3. 使う

`geometry_uid` は全体でも先頭数桁だけでもよい（例: `2f7a041a`）。
複数件ヒットしたら `matches` を見て選び直す。

In [ ]:
# ここを書き換えて実行する（geometry_uid の全体 or 先頭数桁）
query = "2f7a041a"

matches = find_records(query)
print(f"{len(matches)} 件ヒット")
for r in matches:
    print(" ", r.geometry_uid, r.dataset_id, r.institution, r.patient_id, r.file)

In [ ]:
# 1件に絞れたら表示する（複数ヒットした場合は matches[i] で選ぶ）
_ = show_case(matches[0])

## 4. （補足）dataset_id だけで一覧してから選ぶ

`geometry_uid` が分からず、データセット単位でざっと眺めたいときに使う。

In [ ]:
candidates = find_records(None, dataset_id="ANN_EIRLPRJ_1298")
print(f"{len(candidates)} 件")
for r in candidates[:20]:
    print(r.short_uid, r.patient_id, r.file, r.label_lines())